# Ch. 3 · TP2 Classification, ACP et clustering

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/benlahmar/IASciences/blob/main/notebooks/ch03-tp2-classification-clustering.ipynb)

**IA pour les sciences — M1.** Analyses chimiques de vins (jeu réel Wine) : classifier, projeter, regrouper, et comparer.

**Durée indicative :** 3 h

**Objectifs**

- Construire un pipeline de classification sans fuite et lire un rapport de classification.
- Exploiter les probabilités prédites et le choix du seuil.
- Interpréter une ACP (variance expliquée, contributions).
- Choisir le nombre de clusters (coude, silhouette) et comparer aux vraies classes (ARI).
- Démontrer l’effet de la standardisation sur K-means.

**Prérequis :** TP1 du chapitre 3.

> **Mode d’emploi.** Exécutez les cellules dans l’ordre. Les cellules marquées **✏️ Exercice** contiennent des `...` à compléter : une cellule non complétée lève une erreur, c’est normal. Les corrigés sont repliés sous **« Voir le corrigé »** : cherchez d’abord ! Les cellules suivantes ne dépendent jamais du résultat d’un exercice.

## 1. Le jeu Wine : 13 analyses chimiques, 3 cultivars

178 vins d’une même région italienne, issus de **trois cultivars** ; 13 mesures de laboratoire
(alcool, acide malique, magnésium, phénols, flavonoïdes, intensité de couleur, proline…). Les échelles sont très différentes
(proline ~ 1000, autres ~ 1) : un point crucial pour les méthodes à distance.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
X, y = wine.data, wine.target
print(X.shape, '— effectifs par cultivar :', y.value_counts().sort_index().to_dict())
display(X.describe().T[['mean', 'std', 'min', 'max']].round(2))

## 2. Classification supervisée sans fuite

Le `Pipeline` garantit que la standardisation est **apprise sur l’entraînement** à chaque ajustement (et dans chaque pli de validation croisée).
La régression logistique multiclasse calcule pour chaque classe un score linéaire $z_k = w_k^\top x + b_k$,
transformé en probabilités par la **softmax** $p_k = e^{z_k}/\sum_j e^{z_j}$.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.25, stratify=y, random_state=42)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
for name, m in [('Baseline', DummyClassifier()), ('Logistique', clf)]:
    s = cross_val_score(m, X_tr, y_tr, cv=cv, scoring='balanced_accuracy')
    print(f'{name:10s} exactitude équilibrée CV = {s.mean():.3f} ± {s.std():.3f}')

clf.fit(X_tr, y_tr)
pred = clf.predict(X_te)
print(classification_report(y_te, pred, target_names=wine.target_names))
ConfusionMatrixDisplay.from_predictions(y_te, pred, display_labels=wine.target_names)
plt.title('Matrice de confusion — test'); plt.show()

## 3. Au-delà de l’étiquette : les probabilités

`predict_proba` donne la confiance du modèle. Les vins prédits avec une faible probabilité maximale sont ceux à **faire vérifier**
par un expert : c’est souvent plus utile qu’une exactitude globale.

In [ ]:
proba = clf.predict_proba(X_te)
confiance = proba.max(axis=1)
ordre = np.argsort(confiance)[:5]
print('Les 5 prédictions les moins sûres :')
for i in ordre:
    print(f'  vrai={wine.target_names[y_te.iloc[i]]:8s} prédit={wine.target_names[pred[i]]:8s} '
          f'probabilités={proba[i].round(2)}')
plt.hist(confiance, bins=15, edgecolor='white')
plt.xlabel('Probabilité de la classe prédite'); plt.ylabel('Nombre de vins'); plt.show()

## 4. ACP : résumer 13 variables en quelques axes

L’ACP cherche les directions orthogonales de **variance maximale** des données standardisées.
La part de variance expliquée indique combien d’information chaque axe conserve ; les *loadings* indiquent quelles variables y contribuent.
Partie **exploratoire** : réalisée ici sur tout le jeu, elle ne sert pas à évaluer un modèle prédictif.

In [ ]:
from sklearn.decomposition import PCA

X_scaled = StandardScaler().fit_transform(X)
pca = PCA().fit(X_scaled)
var = pca.explained_variance_ratio_
print('Variance expliquée par les 4 premiers axes :', var[:4].round(3), '— cumul :', var[:4].cumsum().round(3))
Z = pca.transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(range(1, 14), var, label='par axe')
axes[0].plot(range(1, 14), var.cumsum(), 'o-', color='C1', label='cumulée')
axes[0].set(xlabel='Composante', ylabel='Part de variance', title='Éboulis des valeurs propres'); axes[0].legend()
for k, nom in enumerate(wine.target_names):
    axes[1].scatter(Z[y == k, 0], Z[y == k, 1], label=nom, alpha=.75)
axes[1].set(xlabel=f'CP1 ({var[0]:.0%})', ylabel=f'CP2 ({var[1]:.0%})', title='Projection ACP colorée par cultivar')
axes[1].legend(); plt.tight_layout(); plt.show()

loadings = pd.DataFrame(pca.components_[:2].T, index=X.columns, columns=['CP1', 'CP2'])
print(loadings.round(2).sort_values('CP1'))

## 5. K-means : regrouper sans étiquettes

K-means minimise l’inertie intra-groupe $\sum_k\sum_{i\in C_k}\|x_i-\mu_k\|^2$. Le nombre $K$ doit être **choisi** :
la méthode du coude (inertie) et le **coefficient de silhouette** (cohésion vs séparation, entre −1 et 1) aident à décider.
L’**ARI** (*adjusted Rand index*) compare une partition aux vraies classes : 0 = hasard, 1 = accord parfait, quel que soit le numéro des clusters.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

inerties, silhouettes = [], []
K = range(2, 9)
for k in K:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_scaled)
    inerties.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(K, inerties, 'o-'); axes[0].set(xlabel='K', ylabel='Inertie', title='Méthode du coude')
axes[1].plot(K, silhouettes, 'o-', color='C2'); axes[1].set(xlabel='K', ylabel='Silhouette', title='Silhouette moyenne')
plt.tight_layout(); plt.show()

clusters = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X_scaled)
print('ARI (K=3, standardisé) :', round(adjusted_rand_score(y, clusters), 3))
print(pd.crosstab(pd.Series(y, name='cultivar'), pd.Series(clusters, name='cluster')))

### ✏️ Exercice 1 — Démonstration : K-means sans standardisation

Relancez K-means (K=3) sur les données **brutes** `X`. Calculez l’ARI. Quelle variable domine la distance euclidienne ? (indice : comparez les écarts-types des colonnes).

In [ ]:
clusters_bruts = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(...)
print('ARI sans standardisation :', round(adjusted_rand_score(y, clusters_bruts), 3))
print(X.std().sort_values(ascending=False).head(3))

<details><summary><b>Voir le corrigé</b></summary>

```python
clusters_bruts = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X)
print('ARI sans standardisation :', round(adjusted_rand_score(y, clusters_bruts), 3))
print(X.std().sort_values(ascending=False).head(3))
```

La proline (écart-type ≈ 315) écrase les autres variables (écarts-types ≈ 0,1 à 10) : les clusters sont presque uniquement des tranches de proline et l’ARI chute. Le choix d’échelle est une **décision scientifique** à justifier.

</details>

### ✏️ Exercice 2 — Classifier avec deux composantes seulement

Construisez un pipeline `StandardScaler → PCA(n_components=2) → LogisticRegression` et comparez son exactitude équilibrée en validation croisée au modèle complet. Pourquoi l’ACP doit-elle être **dans** le pipeline ici ?

In [ ]:
pipe_pca = make_pipeline(StandardScaler(), PCA(n_components=...), LogisticRegression(max_iter=2000))
s = cross_val_score(pipe_pca, ..., ..., cv=cv, scoring='balanced_accuracy')
print(f'Logistique sur 2 CP : {s.mean():.3f} ± {s.std():.3f}')

<details><summary><b>Voir le corrigé</b></summary>

```python
pipe_pca = make_pipeline(StandardScaler(), PCA(n_components=2), LogisticRegression(max_iter=2000))
s = cross_val_score(pipe_pca, X_tr, y_tr, cv=cv, scoring='balanced_accuracy')
print(f'Logistique sur 2 CP : {s.mean():.3f} ± {s.std():.3f}')
```

Deux axes suffisent presque : les cultivars diffèrent selon peu de directions chimiques. L’ACP est une étape **apprise** (moyennes, axes) : pour une évaluation prédictive, elle doit être réapprise dans chaque pli, donc placée dans le pipeline.

</details>

## Interprétation scientifique (à rédiger)

1. Quelles classes sont confondues ? Les probabilités confirment-elles ces confusions ?
2. Quelles variables chimiques structurent les deux premiers axes de l’ACP ?
3. Le nombre de clusters suggéré par la silhouette coïncide-t-il avec les 3 cultivars ? Est-ce une « découverte » ?
4. Quelles données faudrait-il pour affirmer que le modèle reconnaît un cultivar dans une autre région ou un autre millésime ?